<a href="https://colab.research.google.com/github/oscarabrilarranz/Pruebas/blob/main/Top5SP500.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from io import StringIO


def leer_tabla_wikipedia(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/125.0 Safari/537.36"
        )
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return pd.read_html(StringIO(response.text))


def obtener_sp500():
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    tabla = leer_tabla_wikipedia(url)[0]

    tickers = [str(t).replace(".", "-").strip() for t in tabla["Symbol"].tolist()]
    nombres = tabla["Security"].tolist()

    empresas = dict(zip(tickers, nombres))

    print(f"S&P 500 cargado: {len(tickers)} acciones")
    return tickers, empresas


def obtener_datos_fundamentales(ticker):
    try:
        info = yf.Ticker(ticker).get_info()

        per = info.get("trailingPE") or info.get("forwardPE")

        deuda_total = info.get("totalDebt")
        equity = info.get("totalStockholderEquity")

        ratio_deuda = None
        if deuda_total is not None and equity not in (None, 0):
            ratio_deuda = deuda_total / equity

        beneficio = info.get("netIncomeToCommon") or info.get("netIncome")

        return {
            "PER": round(per, 2) if per is not None else None,
            "Ratio deuda": round(ratio_deuda, 2) if ratio_deuda is not None else None,
            "Beneficio neto": beneficio
        }

    except Exception:
        return {
            "PER": None,
            "Ratio deuda": None,
            "Beneficio neto": None
        }


def analizar_sp500(tickers, empresas):
    print("\nDescargando precios del S&P 500...")

    datos = yf.download(
        tickers,
        period="2d",
        interval="1d",
        auto_adjust=True,
        group_by="ticker",
        progress=False,
        threads=True
    )

    resultados = []

    for ticker in tickers:
        try:
            cierre_ayer = datos[ticker]["Close"].iloc[-2]
            cierre_hoy = datos[ticker]["Close"].iloc[-1]

            variacion = ((cierre_hoy - cierre_ayer) / cierre_ayer) * 100

            resultados.append({
                "Empresa": empresas.get(ticker, ""),
                "Ticker": ticker,
                "Cierre ayer": round(cierre_ayer, 2),
                "Cierre hoy": round(cierre_hoy, 2),
                "Variación %": round(variacion, 2)
            })

        except Exception:
            continue

    df = pd.DataFrame(resultados)

    if df.empty:
        return df

    top_caidas = df.sort_values(by="Variación %", ascending=True).head(5)
    top_subidas = df.sort_values(by="Variación %", ascending=False).head(5)

    resultado = pd.concat([top_caidas, top_subidas], ignore_index=True)
    resultado["Tipo"] = ["Mayor caída"] * len(top_caidas) + ["Mayor subida"] * len(top_subidas)

    print("Obteniendo PER, ratio de deuda y beneficio neto...")

    fundamentales = resultado["Ticker"].apply(obtener_datos_fundamentales)
    fundamentales_df = pd.DataFrame(fundamentales.tolist())

    resultado = pd.concat(
        [resultado.reset_index(drop=True), fundamentales_df],
        axis=1
    )

    columnas = [
        "Tipo",
        "Empresa",
        "Ticker",
        "Cierre ayer",
        "Cierre hoy",
        "Variación %",
        "PER",
        "Ratio deuda",
        "Beneficio neto"
    ]

    return resultado[columnas]


def main():
    tickers, empresas = obtener_sp500()
    resultado = analizar_sp500(tickers, empresas)

    print("\n==============================")
    print("S&P 500 - TOP 5 CAÍDAS Y SUBIDAS")
    print("==============================\n")

    if resultado.empty:
        print("No se pudieron obtener datos.")
        return

    print(resultado.to_string(index=False))

    nombre_csv = "sp500_top5_caidas_subidas.csv"
    resultado.to_csv(nombre_csv, index=False)

    print(f"\nCSV generado: {nombre_csv}")


if __name__ == "__main__":
    main()

S&P 500 cargado: 503 acciones

Descargando precios del S&P 500...
Obteniendo PER, ratio de deuda y beneficio neto...

S&P 500 - TOP 5 CAÍDAS Y SUBIDAS

        Tipo                   Empresa Ticker  Cierre ayer  Cierre hoy  Variación %    PER Ratio deuda  Beneficio neto
 Mayor caída Regeneron Pharmaceuticals   REGN       698.25      629.68        -9.82  15.36        None      4423399936
 Mayor caída                  Lumentum   LITE       970.70      884.98        -8.83 154.45        None       439000000
 Mayor caída                    Vertiv    VRT       370.94      339.73        -8.41  85.57        None      1558400000
 Mayor caída       Comfort Systems USA    FIX      1992.74     1854.43        -6.94  53.49        None      1223646976
 Mayor caída              Corning Inc.    GLW       191.81      178.55        -6.91  86.26        None      1810000000
Mayor subida           Dominion Energy      D        61.73       67.56         9.44  19.93        None      2924000000
Mayor subida   